# 07 — Authentication: Bearer, OAuth2, and Signed Webhooks

## Why this notebook exists

Every endpoint and webhook we've built so far (notebooks 02–06) trusts anyone who can reach the URL. That's fine on localhost; it's a serious problem anywhere else.

A2A leans on **OpenAPI's auth model**: the Agent Card declares one or more **`securitySchemes`** (named definitions of *how* auth works) and a **`security`** list (which combinations are required to call this agent). A client reads the card, picks a scheme it can satisfy, and attaches the right credentials.

This notebook walks three real schemes — bearer token, OAuth2 client credentials, and a signed webhook — using the same researcher we've been building, but now with auth enforced.

> *Targets A2A spec v0.3.0. We use HMAC-SHA256 in a header for the signed webhook because it teaches the concept with no extra dependencies; production A2A typically uses JWS/JWT for the same purpose.*

## What you'll learn

- How A2A's `securitySchemes` / `security` fields work and how they map onto OpenAPI's auth model.
- The shape of a **bearer-token** scheme: declared in the card, attached as `Authorization: Bearer …`, validated server-side.
- The shape of an **OAuth2 client-credentials** flow: client posts `client_id` / `client_secret` to a token endpoint, receives an `access_token`, and uses it as a bearer.
- How to **sign** a webhook callback with HMAC-SHA256 in an `X-A2A-Signature` header, and how the receiver verifies it.
- Why "no auth" is itself a security choice that should be declared, not implied.
- The path from these primitives to production-grade A2A auth (JWT, mTLS, real OAuth flows).

## 1. Setup

Same helpers as previous notebooks, plus `hmac` and `hashlib` for webhook signing.

In [ ]:
import hashlib
import hmac
import json
import threading
import time
import uuid
from datetime import datetime, timezone

import httpx
import uvicorn
from fastapi import FastAPI, HTTPException, Request, Header
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")
    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


print("Setup OK")

## 2. Auth Lives in the Agent Card

Two top-level fields on the Agent Card carry all of A2A's auth contract:

| Field | Shape | Purpose |
|---|---|---|
| `securitySchemes` | `{ [name: string]: SecurityScheme }` | Named definitions of authentication methods this agent supports |
| `security` | `[{ [name: string]: string[] }]` | List of acceptable scheme combinations (logical OR between list entries, logical AND within each entry) |

A **`SecurityScheme`** mirrors OpenAPI's:

- `type: "http"` with `scheme: "bearer"` — opaque bearer tokens.
- `type: "oauth2"` with `flows: {...}` — OAuth2 flows (we use `clientCredentials`).
- `type: "apiKey"` with `name: ...` and `in: "header" | "query" | "cookie"` — API keys.
- `type: "mutualTLS"` — client certificates.
- `type: "openIdConnect"` — OIDC.

A typical card has one or two schemes and a single-element `security` list. Multiple list entries express "any of these is acceptable" (e.g., bearer OR oauth2). Multiple keys in one entry express "all of these required together" (e.g., bearer AND a separate API key for tenant routing).

For each demo below we'll show the Agent Card snippet that declares the scheme alongside the code that enforces it.

## 3. Pattern 0: No Auth (Explicitly)

Notebooks 02–06 quietly assumed no auth. The Agent Card was silent on `securitySchemes` and `security`, leaving the empty defaults from notebook 02. That's not wrong — it's the *unauthenticated* default — but it's worth making explicit so a client doesn't have to guess.

```json
{
  "securitySchemes": {},
  "security": []
}
```

An empty `security` list means *"no scheme is required."* Clients should still send the headers they consider polite (e.g., `User-Agent`), but no `Authorization` is needed.